# Machine learning - Model regresji logistycznej

### Źródło danych: Kaggle. https://www.kaggle.com/datasets/shibumohapatra/book-my-show

Firma Book-My-Show planuje wprowadzenie reklam na swojej stronie, jednak chce zadbać o bezpieczeństwo użytkowników i unikać wyświetlania reklam prowadzących do złośliwych linków. Celem projektu jest analiza URL pod kątem wykrywania phishingowych (złośliwych) linków w kontekście systemu BookMyShow. Analiza pozwoli na automatyczne klasyfikowanie adresów URL jako:

1 (Legitimate) – bezpieczna strona

0 (Phishing) – potencjalnie złośliwa strona

-1 (Suspicious) – podejrzana strona, której nie można jednoznacznie sklasyfikować

`Result` - kolumna, wartość której będę przewidywała.

##Struktura danych

Zbiór zawiera 11 000 próbek URL z 32 cechami opisującymi właściwości techniczne adresów. **Wartość każdej cechy może być -1,0,1(podejrzana, potencjalnie złośliwa oraz bezpieczna strond odpowiednio)**.

W zbiorze przedstawione takie cechy jak:

`index`

`having_IPhaving_IP_Address` - Czy URL zawiera adres IP zamiast nazwy domeny

`URLURL_Length` - Długość całego URL (liczba znaków)

`Shortining_Service` - Czy URL korzysta z usługi skracania linków (np. bit.ly)

`having_At_Symbol` - Obecność znaku '@' w URL – znak '@' może powodować, że przeglądarka ignoruje część adresu, co jest podejrzane

`double_slash_redirecting` - Występowanie podwójnego ukośnika '//' w nietypowym miejscu URL

`Prefix_Suffix` - Obecność myślnika '-' w nazwie domeny – phishingowe domeny często używają myślników do podszywania się

`having_Sub_Domain` - Liczba poddomen w URL – duża liczba poddomen może być podejrzana

`SSLfinal_State` - Status certyfikatu SSL/TLS strony (np. ważny, wygasły, brak)

`Domain_registeration_length` - Długość rejestracji domeny w dniach

`Favicon` - Czy favicon (ikona strony) pochodzi z tej samej domeny

`port` - Numer portu używanego przez serwer

`HTTPS_token` - Obecność słowa "https" w nazwie domeny lub URL

`Request_URL` - Procent zasobów (np. obrazów, skryptów) ładowanych z zewnętrznych domen

`URL_of_Anchor` - Procent linków (anchor tags) prowadzących do zewnętrznych lub podejrzanych domen

`Links_in_tags` - Procent linków w tagach HTML

`SFH` - Status pola formularza (Server Form Handler) – czy formularz wysyła dane do zaufanej domeny

`Submitting_to_email` - Czy formularz na stronie wysyła dane bezpośrednio na adres e-mail

`Abnormal_URL` - Wskaźnik niestandardowej lub podejrzanej struktury URL

`Redirect` - Liczba przekierowań URL

`on_mouseover` - Obecność zdarzeń JavaScript onmouseover zmieniających adres lub zachowanie

`RightClick` - Blokada kliknięcia prawym przyciskiem myszy na stronie

`popUpWidnow` - Obecność wyskakujących okienek (pop-up)

`Iframe` - Obecność ramek iframe osadzających zewnętrzne strony

`age_of_domain` - Wiek domeny

`DNSRecord` - Czy istnieje poprawny rekord DNS dla domeny

`web_traffic` - Ruch na stronie

`Page_Rank` - Ranking strony w wyszukiwarkach

`Google_Index` - Czy strona jest zaindeksowana przez Google

`Links_pointing_to_page` - Liczba linków prowadzących do strony

`Statistical_report` - Informacje z baz danych lub raportów statystycznych dotyczące reputacji strony

`Result` - Etykieta klasyfikacji URL: phishing lub bezpieczna (wartość docelowa do przewidywania)



In [ ]:
#Importuję niezbędne biblioteki
import pandas as pd
import sklearn
import sklearn.linear_model as sk
from sklearn.model_selection import train_test_split

#Podłączam do mojego Google Dysku
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Wgrywam dane z pliku
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/ML-projekt zaliczeniowy/dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11055 entries, 0 to 11054
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   index                        11055 non-null  object
 1   having_IPhaving_IP_Address   11055 non-null  int64 
 2   URLURL_Length                11055 non-null  int64 
 3   Shortining_Service           11055 non-null  int64 
 4   having_At_Symbol             11055 non-null  int64 
 5   double_slash_redirecting     11055 non-null  int64 
 6   Prefix_Suffix                11055 non-null  int64 
 7   having_Sub_Domain            11055 non-null  int64 
 8   SSLfinal_State               11055 non-null  int64 
 9   Domain_registeration_length  11055 non-null  int64 
 10  Favicon                      11055 non-null  int64 
 11  port                         11055 non-null  int64 
 12  HTTPS_token                  11055 non-null  int64 
 13  Request_URL                  11

In [ ]:
df.head() #Sprawdzam wygląd tabelki przed zmianami

,index,having_IPhaving_IP_Address,URLURL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,1,0,1,1,1,0,0,0,0,0,...,1,1,0,0,0,0,1,1,0,0
1,2,1,1,1,1,1,0,-1,1,0,...,1,1,0,0,-1,0,1,1,1,0
2,3,1,-1,1,1,1,0,0,0,0,...,1,1,1,0,1,0,1,-1,0,0
3,4,1,-1,1,1,1,0,0,0,1,...,1,1,0,0,1,0,1,0,1,0
4,5,1,-1,0,1,1,0,1,1,0,...,0,1,0,0,-1,0,1,1,1,1


In [ ]:
df = df.drop(["index"], axis=1) #usuwam kolumnę index, ponieważ nie niesie informacji merytorycznej o danych
df.isna().sum() #sprawdzam wartości brakujące w kolumnach

,0
having_IPhaving_IP_Address,0
URLURL_Length,0
Shortining_Service,0
having_At_Symbol,0
double_slash_redirecting,0
Prefix_Suffix,0
having_Sub_Domain,0
SSLfinal_State,0
Domain_registeration_length,0
Favicon,0


Zdecydowałam się pozostawić wszystkie kolumny bez zmian, ponieważ dane są kompletne, nie zawierają brakujących wartości, a cechy mają dobrze zdefiniowane wartości (-1, 0, 1), które są odpowiednie do analizy i modelowania przy użyciu regresji logistycznej. Dzięki temu planuję zachować pełną informację potrzebną do skutecznej klasyfikacji URL.

In [ ]:
df.head() # dane po przygotowaniu

,having_IPhaving_IP_Address,URLURL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,...,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report,Result
0,0,1,1,1,0,0,0,0,0,1,...,1,1,0,0,0,0,1,1,0,0
1,1,1,1,1,1,0,-1,1,0,1,...,1,1,0,0,-1,0,1,1,1,0
2,1,-1,1,1,1,0,0,0,0,1,...,1,1,1,0,1,0,1,-1,0,0
3,1,-1,1,1,1,0,0,0,1,1,...,1,1,0,0,1,0,1,0,1,0
4,1,-1,0,1,1,0,1,1,0,1,...,0,1,0,0,-1,0,1,1,1,1


###Podział danych

Podzieliłam zbiór danych na 70% danych treningowych i 30% testowych.

In [ ]:
train, test = train_test_split(df, test_size=0.3, stratify=df["Result"])
print(train.shape, test.shape) #rozmiar zbioru danych treningowych i testowych

(7738, 31) (3317, 31)


Oddzieliłam kolumnę dotyczącą wyników od pozostałych kolumn.

X = dane treningowe

Y = odpowiedzi

In [ ]:
train_y=train["Result"]
train_x=train.drop(["Result"], axis=1)

# W danych testowych też oddzieliłam kolumnę dot. wyników
test_y=test["Result"]
test_x=test.drop(["Result"], axis=1)

print("Train:", train_y.value_counts())
print("Test:", test_y.value_counts())

Train: Result
1    4310
0    3428
Name: count, dtype: int64
Test: Result
1    1847
0    1470
Name: count, dtype: int64


### Budowanie modelu

In [ ]:
model = sk.LogisticRegression()
model.fit(train_x,train_y) #proces dopasowania modelu

# Efekt dopasowania
print("Współczynniki :", model.coef_[0])
print("Stała regresji :", model.intercept_[0])

Współczynniki : [ 1.55015539  0.11134977 -1.4363749   0.6919167   0.33055652  4.96489456
  0.84980981  3.65844497 -0.50182897 -0.34362793 -0.12519474 -0.61668543
  0.49097796  0.0777821  -0.28040854  0.39016179  0.1831887  -0.71075979
 -0.43639459  0.36295626  0.16930724 -0.10138629 -0.18475775  0.04101394
  1.67717476  1.21469837  0.44271322  1.1008837   0.77050295  0.70199618]
Stała regresji : -4.932992117666904


In [ ]:
# Generuję przewidywania (1,0) dla każdej z wartości testowych
rezultaty = model.predict(test_x)
rezultaty = list(rezultaty)
# Zapisałam faktyczne odpowiedzi do zwykłej listy
odpowiedzi= list(test_y)

# Zapisałam wyniki w liście: (oczekiwany_wynik, otrzymany_wynik)
testy=[]
for i in range(len(rezultaty)):
  testy.append((odpowiedzi[i],rezultaty[i]))

In [ ]:
total=len(testy)

# Wartość prawdziwie dodatnia - bezpieczna strona i tak przewidziałam
tp = testy.count((1,1))
# Wartość prawdziwie ujemna - podejrzana strona i tak przewidziałam
tn = testy.count((0,0))
# Wartość fałszywie dodatnia - podejrzana strona, a przewidziałam, że bezpieczna
fp = testy.count((0,1))
# Wartość fałszywnie ujemna - bezpieczna strona, a przewidziałam, że jest niebezpieczna
fn = testy.count((1,0))

print(f"TP = {tp}, TN = {tn}, FP = {fp}, Fn = {fn}, w sumie {total} przypadków")

In [ ]:
# Obliczam wskażniki: dokładność, precyzja, czułość, swoistość, f-score, zbalansowana dokładność
accuracy=(tp+tn)/total # ile przypadków zostały poprawnie przewidziane

precision=tp/(tp+fp) # jaki procent stron oznaczonych jako bezpieczne faktycznie były bezpieczne

sensitivity=tp/(tp+fn) # jaki procent faktycznie bezpiecznych stron został wykryty

specificity=tn/(tn+fp)

balanced_accuracy=(sensitivity+specificity)/2

fscore=(2*precision*sensitivity)/(precision+sensitivity)

# Użyte formatowanie liczb do dwóch cyfr po przecinku
print(f"Dokładność:\t\t{accuracy}\t{'%.02f' % (accuracy*100)}% stron udało się poprawnie zaklasyfikować")
print(f"Precyzja:\t\t{precision}\t{'%.02f' % (precision*100)}% stron, których zakwalifikowałam jako bezpieczne, faktycznie były bezpieczne")
print()
print(f"Czułość:\t\t{sensitivity}\t{'%.02f' % (sensitivity*100)}% bezpiecznych stron zaklasyfikowano poprawnie")
print(f"Swoistość:\t\t{specificity}\t{'%.02f' % (specificity*100)}% podejrzanych stron zaklasyfikowano poprawnie")
print(f"Zbalansowana dokładność:{balanced_accuracy}\t{'%.02f' % (balanced_accuracy*100)}% to wskaźnik uśredniający czułość i swoistość")
print(f"F-score:\t\t{fscore}\t{'%.02f' % (fscore*100)}% to wskaźnik balansujący wyniki precyzji i czułości")

Dokładność:		0.9083509195055773	90.84% stron udało się poprawnie zaklasyfikować
Precyzja:		0.9172525689561926	91.73% stron, których zakwalifikowałam jako bezpieczne, faktycznie były bezpieczne

Czułość:		0.918245804006497	91.82% bezpiecznych stron zaklasyfikowano poprawnie
Swoistość:		0.8959183673469387	89.59% podejrzanych stron zaklasyfikowano poprawnie
Zbalansowana dokładność:0.9070820856767179	90.71% to wskaźnik uśredniający czułość i swoistość
F-score:		0.9177489177489179	91.77% to wskaźnik balansujący wyniki precyzji i czułości


### Uśrednienie wyników

In [ ]:
def regresja(df):
  train, test = train_test_split(df, test_size=0.3, stratify=df["Result"])
  train_y=train["Result"]
  train_x=train.drop(["Result"], axis=1)
  test_y=test["Result"]
  test_x=test.drop(["Result"], axis=1)

  model = sk.LogisticRegression()
  model.fit(train_x,train_y)

  rezultaty = list(model.predict(test_x))
  odpowiedzi= list(test_y)
  testy=[]
  for i in range(len(rezultaty)):
    testy.append((odpowiedzi[i],rezultaty[i]))

  tp = testy.count((1,1))
  tn = testy.count((0,0))
  fp = testy.count((0,1))
  fn = testy.count((1,0))
  total=(len(testy))

  accuracy=(tp+tn)/total
  precision=tp/(tp+fp)
  sensitivity=tp/(tp+fn)
  specificity=tn/(tn+fp)
  return [accuracy,precision,sensitivity,specificity]

n=100
wynik=[0,0,0,0]
for i in range(n):
  results=regresja(df)
  for i,v in enumerate(results):
    wynik[i]+=v

# uśrednianie
wynik=[w/n for w in wynik]
print(f'''accuracy: {wynik[0]}
precision: {wynik[1]}
sensitivity: {wynik[2]}
specificity: {wynik[3]}''')

accuracy: 0.9116792282182693
precision: 0.9171794407946736
sensitivity: 0.9249539794260965
specificity: 0.8949999999999996


## Podsumowanie

Uważam ,że wyniki uzyskane dla modelu regresji logistycznej są bardzo dobre i wskazują na wysoką skuteczność klasyfikatora:

Dokładność (accuracy) na poziomie około 91% oznacza, że model poprawnie klasyfikuje zdecydowaną większość stron.

Precyzja (91,7%) pokazuje, że większość stron zaklasyfikowanych jako bezpieczne rzeczywiście jest bezpieczna, co minimalizuje liczbę fałszywie pozytywnych wyników.

Czułość (sensitivity, 91,8%) świadczy o tym, że model wykrywa prawie wszystkie bezpieczne strony, nie pomijając ich w klasyfikacji.

Swoistość (specificity, 89,6%) oznacza, że model skutecznie rozpoznaje podejrzane strony, nie myląc ich z bezpiecznymi.

Zbalansowana dokładność (90,7%) potwierdza, że model zachowuje równowagę pomiędzy wykrywaniem obu klas.

F-score (91,8%) dodatkowo potwierdza, że model dobrze radzi sobie z kompromisem pomiędzy precyzją a czułością.

### Skuteczność modelu

Powyższe wskaźniki jednoznacznie sugerują, że model jest bardzo skuteczny w klasyfikowaniu URL-i jako bezpieczne lub phishingowe. Wysokie wartości wszystkich kluczowych miar świadczą o tym, że model nie tylko poprawnie rozpoznaje większość przypadków, ale także dobrze radzi sobie zarówno z wykrywaniem zagrożeń, jak i unikaniem fałszywych alarmów.

### Warianty badania

W analizie wykorzystano wszystkie dostępne kolumny cech, ponieważ:

- Dane były kompletne, nie zawierały wartości brakujących ani oczywistych cech zbędnych.

- Wszystkie cechy mają potencjalną wartość informacyjną dla wykrywania phishingu, a ich usuwanie mogłoby prowadzić do utraty ważnych sygnałów.

Nie testowano wariantów z usuwaniem poszczególnych cech, ponieważ zestaw został przygotowany w sposób umożliwiający bezpośrednie zastosowanie do klasyfikacji, a każda z cech wnosi unikalną informację o strukturze lub zachowaniu URL.

Można by przetestować model po usunięciu wybranych cech, aby ocenić wpływ na skuteczność. Warto też zbadać korelacje między 32 cechami i rozważyć ich grupowanie(podział na faktory), choć mogłoby to utrudnić interpretację wpływu poszczególnych czynników.
